# Notebook 02 — Modeling & policy levers

**Purpose:** Model AI adoption and downstream outcomes, then export compact “policy lever” tables for Tableau.

**Input (generated by Notebook 01):**
- `data/processed/so2025_tableau_clean_final.csv`

**Outputs:**
- `outputs/tables/tableau_policy_lever_impacts_strategic.csv`
- `outputs/tables/tableau_policy_lever_impacts.csv` (intermediate)


In [13]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [5]:
# Load the cleaned, slimmed dataset you already prepared for Tableau
df = pd.read_csv("data/processed/so2025_tableau_clean_final.csv")

print(df.shape)
df.head()


(49123, 58)


,ResponseId,Country,Industry,EdLevel,Employment,ICorPM,Pay_USD,LogPay,AISelect,LearnCodeAI,AIAgents,AI_IntensityScore,HighAI,DevType,DevType_bucket,OrgSize_clean,RemoteWork_clean,ExpBand,WorkExp_num,YearsCode_num,NumLangs,NumDBs,NumClouds,NumWeb,SQL_any,BI_any,ToolCountWork,ToolCountPersonal,JobSat,SOVisitFreq,SOPartFreq,AISent,AIAcc,AIComplex,AIThreat,SOFriction,NewRole,MainBranch_clean,MainBranch_sort,SOAccount_clean,HasSOAccount,SODuration_clean,SODuration_sort,SODuration_years,SOComm_clean,SOComm_likert,SOComm_sort,EmpAddl_none_of_the_above,EmpAddl_caring_for_dependents_children_elder,EmpAddl_engaged_in_paid_work_20_29_hours_per,EmpAddl_volunteering_regularly,EmpAddl_attending_school_full_time,EmpAddl_engaged_in_paid_work_less_than_10_ho,EmpAddl_attending_school_part_time,EmpAddl_engaged_in_paid_work_10_19_hours_per,EmpAddl_transitioning_to_retirement_graduall,EmpAddl_count,EmpAddl_Other
0,1,Ukraine,Fintech,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,People manager,61256.0,11.022817,"Yes, I use AI tools monthly or infrequently","Yes, I learned how to use AI-enabled tools for...","Yes, I use AI agents at work monthly or infreq...",3.6,0,"Developer, mobile",Mobile,10–49,Remote,6–10,8.0,14.0,3,2,5,0,1,0,7.0,3.0,10.0,A few times per week,I have never participated in Q&A on Stack Over...,Indifferent,Neither trust nor distrust,Bad at handling complex tasks,I'm not sure,"Rarely, almost never",I have neither consider or transitioned into a...,Pro developer,1,Yes,1.0,Between 5 and 10 years,4.0,7.5,Neutral,3.0,3.0,0,1,0,0,0,0,0,0,0,1,0
1,2,Netherlands,Retail and Consumer Services,"Associate degree (A.A., A.S., etc.)",Employed,Individual contributor,104413.0,11.556109,"Yes, I use AI tools weekly","Yes, I learned how to use AI-enabled tools for...","No, and I don't plan to",3.0,0,"Developer, back-end",Web/Fullstack,250–999,Hybrid,0–2,2.0,10.0,1,2,7,1,0,0,6.0,5.0,9.0,Multiple times per day,"Infrequently, less than once per year",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,I'm not sure,About half of the time,I have transitioned into a new career and/or i...,Pro developer,1,Not sure,NaN,Between 10 and 15 years,5.0,12.5,"Yes, somewhat",4.0,4.0,0,0,0,0,0,0,0,0,0,0,0
2,3,Ukraine,Software Development,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",NaN,53061.0,10.879197,"Yes, I use AI tools daily","Yes, I learned how to use AI-enabled tools for...","Yes, I use AI agents at work weekly",5.0,1,"Developer, front-end",Web/Fullstack,NaN,NaN,6–10,10.0,12.0,4,3,4,3,1,0,3.0,3.0,8.0,A few times per week,"Infrequently, less than once per year",Favorable,Somewhat trust,Neither good or bad at handling complex tasks,No,About half of the time,I have transitioned into a new career and/or i...,Pro developer,1,Not sure,NaN,Between 5 and 10 years,4.0,7.5,Neutral,3.0,3.0,1,0,0,0,0,0,0,0,0,1,0
3,4,Ukraine,Retail and Consumer Services,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,Individual contributor,36197.0,10.496732,"Yes, I use AI tools weekly","Yes, I learned how to use AI-enabled tools for...","Yes, I use AI agents at work monthly or infreq...",3.4,0,"Developer, back-end",Web/Fullstack,10k+,Remote,3–5,4.0,5.0,3,0,2,1,1,0,NaN,NaN,6.0,A few times per month or weekly,I have never participated in Q&A on Stack Over...,Favorable,Somewhat trust,Bad at handling complex tasks,No,"Rarely, almost never",I have neither consider or transitioned into a...,Pro developer,1,Yes,1.0,Between 3 and 5 years,3.0,4.0,Neutral,3.0,3.0,1,0,0,0,0,0,0,0,0,1,0
4,5,Ukraine,Software Development,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",NaN,60000.0,11.002100,"Yes, I use AI tools weekly","Yes, I learned how to use AI-enabled tools for...","No, and I don't plan to",3.0,0,Engineering manager,Other/Unknown,NaN,NaN,16+,21.0,22.0,14,7,9,5,1,0,4.0,3.0,7.0,A few times per month or weekly,"Infrequently, less than once per year",Favorable,N

In [ ]:
# --- Derived fields for modeling stability ---
# JobSat is already 0-10 in our cleaned extract; keep as numeric
df["JobSat_num"] = pd.to_numeric(df["JobSat"], errors="coerce")

# AIThreat is {Yes, No, I'm not sure}; map to binary (drop unsure)
df["AIThreat_bin"] = df["AIThreat"].map({"Yes": 1, "No": 0})

# SOFriction is ordinal frequency; map to numeric, keep non-AI users as NaN
friction_map = {
    "Rarely, almost never": 0,
    "Less than half of the time": 1,
    "About half of the time": 2,
    "More than half the time": 3
}
df["SOFriction_num"] = df["SOFriction"].map(friction_map)

# Defensive fills for lever fields
if "HasSOAccount" in df.columns:
    df["HasSOAccount"] = df["HasSOAccount"].fillna(0).astype(int)


In [15]:
key_cols = [
    "HighAI", "AI_IntensityScore", "ToolCountWork", "JobSat",
    "AIThreat", "SOFriction", "AISelect", "LearnCodeAI", "AIAgents",
    "AISent", "SOVisitFreq", "SOPartFreq"
]

print(df[key_cols].isna().mean().sort_values(ascending=False))


JobSat               0.457627
ToolCountWork        0.440445
SOFriction           0.373756
AIAgents             0.350813
SOPartFreq           0.344503
SOVisitFreq          0.334731
AISent               0.319382
AISelect             0.314252
AIThreat             0.266311
LearnCodeAI          0.080818
HighAI               0.000000
AI_IntensityScore    0.000000
dtype: float64


In [35]:
# === 1) LearnCodeAI: did they learn AI tools (job or hobby)? ===

learn_ai_yes = [
    "Yes, I learned how to use AI-enabled tools required for my job or to benefit my career",
    "Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies"
]

df["Lever_LearnCodeAI"] = np.where(df["LearnCodeAI"].isin(learn_ai_yes), 1, 0)


# === 2) AISelect: high AI usage (weekly/daily vs others) ===

ai_high = [
    "Yes, I use AI tools daily",
    "Yes, I use AI tools weekly"
]

df["Lever_AIUseHigh"] = np.where(df["AISelect"].isin(ai_high), 1, 0)


# === 3) AIAgents: any agent usage vs not ===

agents_yes = [
    "Yes, I use AI agents at work daily",
    "Yes, I use AI agents at work weekly",
    "Yes, I use AI agents at work monthly or infrequently"
]

df["Lever_AIAgentsAny"] = np.where(df["AIAgents"].isin(agents_yes), 1, 0)


# === 4) Skill breadth levers: NumClouds / NumLangs above median ===

cloud_median = df["NumClouds"].median()
lang_median = df["NumLangs"].median()
print("Cloud median:", cloud_median, " | Lang median:", lang_median)

df["Lever_NumCloudsHigh"] = np.where(df["NumClouds"] >= cloud_median, 1, 0)
df["Lever_NumLangsHigh"] = np.where(df["NumLangs"] >= lang_median, 1, 0)


# === 5) Existing flags: SQL_any, HasSOAccount ===

df["Lever_SQL_any"] = df["SQL_any"].fillna(0).astype(int)
df["Lever_HasSOAccount"] = df["HasSOAccount"].fillna(0).astype(int)


# === 6) SOVisitFreq: high visit frequency (multiple/day + daily + few/wk) ===

so_high = [
    "Multiple times per day",
    "Daily or almost daily",
    "A few times per week"
]
df["Lever_SOVisitHigh"] = np.where(df["SOVisitFreq"].isin(so_high), 1, 0)


# === 7) AISent: positive stance (Favorable / Very favorable) ===

ai_pos = ["Favorable", "Very favorable"]
df["Lever_AISentPos"] = np.where(df["AISent"].isin(ai_pos), 1, 0)


lever_cols = [
    "Lever_LearnCodeAI",
    "Lever_AIUseHigh",
    "Lever_AIAgentsAny",
    "Lever_NumLangsHigh",
    "Lever_SQL_any",
    "Lever_HasSOAccount",
    "Lever_SOVisitHigh",
    "Lever_AISentPos"
]

df[lever_cols].describe()


Cloud median: 0.0  | Lang median: 4.0


,Lever_LearnCodeAI,Lever_AIUseHigh,Lever_AIAgentsAny,Lever_NumLangsHigh,Lever_SQL_any,Lever_HasSOAccount,Lever_SOVisitHigh,Lever_AISentPos
count,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000
mean,0.618692,0.444069,0.200883,0.522525,0.515359,0.543045,0.351343,0.406449
std,0.485713,0.496867,0.400665,0.499497,0.499769,0.498149,0.477395,0.491175
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [45]:
# Final lever set (after diagnostics)
lever_cols = [
    "Lever_LearnCodeAI",
    "Lever_AIUseHigh",
    "Lever_AIAgentsAny",
    "Lever_NumLangsHigh",
    "Lever_SQL_any",
    "Lever_HasSOAccount",
    "Lever_SOVisitHigh",
    "Lever_AISentPos"
]

df[lever_cols].describe()


,Lever_LearnCodeAI,Lever_AIUseHigh,Lever_AIAgentsAny,Lever_NumLangsHigh,Lever_SQL_any,Lever_HasSOAccount,Lever_SOVisitHigh,Lever_AISentPos
count,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000,49123.000000
mean,0.618692,0.444069,0.200883,0.522525,0.515359,0.543045,0.351343,0.406449
std,0.485713,0.496867,0.400665,0.499497,0.499769,0.498149,0.477395,0.491175
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [47]:
# Reduced, stable control set
control_cat_cols = [
    "DevType_bucket",
    "ExpBand",
    "OrgSize_clean",
    "RemoteWork_clean"
]

for c in control_cat_cols:
    df[c] = df[c].astype("category")

# Columns required for modeling
model_cols = (
    ["HighAI", "AI_IntensityScore", "ToolCountWork", "JobSat_num",
     "AIThreat_bin", "SOFriction_num"]
    + lever_cols
    + control_cat_cols
)

df_model = df[model_cols].dropna().copy()
print("Modeling sample shape:", df_model.shape)
df_model.head()


Modeling sample shape: (16409, 18)


,HighAI,AI_IntensityScore,ToolCountWork,JobSat_num,AIThreat_bin,SOFriction_num,Lever_LearnCodeAI,Lever_AIUseHigh,Lever_AIAgentsAny,Lever_NumLangsHigh,Lever_SQL_any,Lever_HasSOAccount,Lever_SOVisitHigh,Lever_AISentPos,DevType_bucket,ExpBand,OrgSize_clean,RemoteWork_clean
0,0,3.6,7.0,10.0,0,1.0,1,0,1,0,1,1,1,0,Mobile,6–10,10–49,Remote
1,0,3.0,6.0,9.0,0,3.0,1,1,0,0,0,0,1,0,Web/Fullstack,0–2,250–999,Hybrid
7,1,4.2,10.0,6.0,1,3.0,1,1,0,1,1,1,0,1,Other/Unknown,16+,10–49,Remote
9,0,0.2,6.0,4.0,0,1.0,0,0,0,0,0,1,1,0,Mobile,6–10,10–49,Remote
10,1,5.3,9.0,10.0,0,3.0,1,1,1,1,1,1,1,1,Web/Fullstack,6–10,50–249,Remote


In [49]:
# Build formula: HighAI ~ Levers + Controls
lever_terms   = " + ".join(lever_cols)
control_terms = " + ".join([f"C({c})" for c in control_cat_cols])

formula_highai = f"HighAI ~ {lever_terms} + {control_terms}"
print("HighAI formula:\n", formula_highai)

# Penalized logistic regression (L1) to avoid singular matrix / separation
logit_highai = smf.logit(formula_highai, data=df_model).fit_regularized(method="l1")
logit_highai.summary()


HighAI formula:
 HighAI ~ Lever_LearnCodeAI + Lever_AIUseHigh + Lever_AIAgentsAny + Lever_NumLangsHigh + Lever_SQL_any + Lever_HasSOAccount + Lever_SOVisitHigh + Lever_AISentPos + C(DevType_bucket) + C(ExpBand) + C(OrgSize_clean) + C(RemoteWork_clean)
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.21298996310487958
            Iterations: 189
            Function evaluations: 190
            Gradient evaluations: 189


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 HighAI   No. Observations:                16409
Model:                          Logit   Df Residuals:                    16382
Method:                           MLE   Df Model:                           26
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                  0.6599
Time:                        17:31:50   Log-Likelihood:                -3495.0
converged:                       True   LL-Null:                       -10276.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                            -13.2730      0.403    -32.907      0.000     -14.064     -12.482
C(DevType_bucket)[T.DevOps/Cloud]     -0.4896      0.202     -2.429      0.015      -0.885      -0.095
C(DevType_bucket)[T.Mobile]           -0.2838      0.197     -1.439      0.150      -0.670       0.103
C(DevType_bucket)[T.Other/Unknown]    -0.3579      0.117     -3.065      0.002      -0.587      -0.129
C(DevType_bucket)[T.Security]          0.0094      0.419      0.022      0.982      -0.812       0.831
C(DevType_bucket)[T.Web/Fullstack]    -0.3976      0.126     -3.151      0.002      -0.645      -0.150
C(ExpBand)[T.11–15]                    0.1089      0.131      0.834      0.405      -0.147       0.365
C(ExpBand)[T.16+]                      0.1647      0.121      1.357      0.175      -0.073       0.403
C(ExpBand)[T.3–5]                      0.1362      0.127      1.069      0.285      -0.113       0.386
C(ExpBand)[T.6–10]                     0.3008      0.123      2.452      0.014       0.060       0.541
C(OrgSize_clean)[T.10k+]              -0.2147      0.167     -1.287      0.198      -0.542       0.112
C(OrgSize_clean)[T.10–49]             -0.2749      0.155     -1.777      0.076      -0.578       0.028
C(OrgSize_clean)[T.1k–4.9k]           -0.3172      0.171     -1.851      0.064      -0.653       0.019
C(OrgSize_clean)[T.250–999]           -0.2439      0.186     -1.314      0.189      -0.608       0.120
C(OrgSize_clean)[T.50–249]            -0.3117      0.163     -1.918      0.055      -0.630       0.007
C(OrgSize_clean)[T.5k–9.9k]           -0.3111      0.204     -1.521      0.128      -0.712       0.090
C(OrgSize_clean)[T.Unknown]           -0.3778      0.300     -1.258      0.208      -0.966       0.211
C(RemoteWork_clean)[T.Onsite]          0.1072      0.100      1.076      0.282      -0.088       0.302
C(RemoteWork_clean)[T.Remote]          0.0246      0.069      0.356      0.722      -0.111       0.160
Lever_LearnCodeAI                      4.8127      0.155     30.971      0.000       4.508       5.117
Lever_AIUseHigh                        5.8719      0.230     25.513      0.000       5.421       6.323
Lever_AIAgentsAny                      4.9774      0.104     47.793      0.000       4.773       5.182
Lever_NumLangsHigh                     0.3431      0.088      3.893      0.000       0.170       0.516
Lever_SQL_any                          0.4109      0.092      4.444      0.000       0.230       0.592
Lever_HasSOAccount                     0.2570      0.092      2.808      0.005       0.078       0.436
Lever_SOVisitHigh                     -0.1030      0.062     -1.669      0.095      -0.224       0.018
Lever_AISentPos                        1.0991      0.092     11.997      0.000       0.920       1.279
======================================================================================================

Possibly complete quasi-separation: A fraction 

In [51]:
formula_tool = f"ToolCountWork ~ HighAI + AI_IntensityScore + {control_terms}"
print("ToolCountWork formula:\n", formula_tool)

ols_tool = smf.ols(formula_tool, data=df_model).fit()
print(ols_tool.summary().tables[1])


ToolCountWork formula:
 ToolCountWork ~ HighAI + AI_IntensityScore + C(DevType_bucket) + C(ExpBand) + C(OrgSize_clean) + C(RemoteWork_clean)
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              5.8898      0.437     13.482      0.000       5.034       6.746
C(DevType_bucket)[T.DevOps/Cloud]      2.2966      0.385      5.972      0.000       1.543       3.050
C(DevType_bucket)[T.Mobile]           -1.0288      0.375     -2.745      0.006      -1.764      -0.294
C(DevType_bucket)[T.Other/Unknown]     0.3143      0.240      1.312      0.190      -0.155       0.784
C(DevType_bucket)[T.Security]          3.5729      0.856      4.174      0.000       1.895       5.251
C(DevType_bucket)[T.Web/Fullstack]    -0.6128      0.256     -2.395      0.017      -1.114      -0.111
C(ExpBand)[T.11–15]                

In [53]:
formula_jobsat = f"JobSat_num ~ HighAI + ToolCountWork + AIThreat_bin + {control_terms}"
print("JobSat formula:\n", formula_jobsat)

ols_jobsat = smf.ols(formula_jobsat, data=df_model).fit()
print(ols_jobsat.summary().tables[1])


JobSat formula:
 JobSat_num ~ HighAI + ToolCountWork + AIThreat_bin + C(DevType_bucket) + C(ExpBand) + C(OrgSize_clean) + C(RemoteWork_clean)
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              7.5005      0.113     66.335      0.000       7.279       7.722
C(DevType_bucket)[T.DevOps/Cloud]     -0.1327      0.101     -1.308      0.191      -0.331       0.066
C(DevType_bucket)[T.Mobile]           -0.0797      0.099     -0.807      0.420      -0.273       0.114
C(DevType_bucket)[T.Other/Unknown]    -0.0748      0.063     -1.186      0.236      -0.198       0.049
C(DevType_bucket)[T.Security]          0.2904      0.226      1.287      0.198      -0.152       0.733
C(DevType_bucket)[T.Web/Fullstack]    -0.2519      0.067     -3.738      0.000      -0.384      -0.120
C(ExpBand)[T.11–15]               

In [55]:
# AIThreat_bin as logistic (also regularized for robustness)
formula_threat = f"AIThreat_bin ~ HighAI + ToolCountWork + {control_terms}"
print("AIThreat formula:\n", formula_threat)

logit_threat = smf.logit(formula_threat, data=df_model).fit_regularized(method="l1")
print(logit_threat.summary())

# SOFriction_num as linear (approx)
formula_sofr = f"SOFriction_num ~ HighAI + ToolCountWork + {control_terms}"
print("SOFriction formula:\n", formula_sofr)

ols_sofr = smf.ols(formula_sofr, data=df_model).fit()
print(ols_sofr.summary().tables[1])


AIThreat formula:
 AIThreat_bin ~ HighAI + ToolCountWork + C(DevType_bucket) + C(ExpBand) + C(OrgSize_clean) + C(RemoteWork_clean)
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.38647724473180084
            Iterations: 150
            Function evaluations: 152
            Gradient evaluations: 150
                           Logit Regression Results                           
Dep. Variable:           AIThreat_bin   No. Observations:                16409
Model:                          Logit   Df Residuals:                    16388
Method:                           MLE   Df Model:                           20
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                 0.01867
Time:                        17:32:25   Log-Likelihood:                -6341.7
converged:                       True   LL-Null:                       -6462.3
Covariance Type:            nonrobust   LLR p-value:                 6.568e-40
                       

In [57]:
def lever_delta_highai(logit_model, data, lever_col):
    """
    For each binary lever, compute the average change in P(HighAI)
    if we toggle that lever from 0 -> 1 for everyone.
    """
    dtmp = data.dropna(subset=[lever_col]).copy()
    if dtmp.empty:
        return np.nan
    
    d0 = dtmp.copy()
    d0[lever_col] = 0
    p0 = logit_model.predict(d0)
    
    d1 = dtmp.copy()
    d1[lever_col] = 1
    p1 = logit_model.predict(d1)
    
    return (p1 - p0).mean()


def marginal_effect_highai(logit_model, data, var_name="HighAI"):
    """
    Approximate average marginal effect of HighAI in a logit model
    using p*(1-p)*beta.
    """
    p = logit_model.predict(data)
    beta = logit_model.params[var_name]
    return beta * np.mean(p * (1 - p))


# Downstream coefficients
beta_highai_tool   = ols_tool.params.get("HighAI", 0.0)
beta_highai_jobsat = ols_jobsat.params.get("HighAI", 0.0)
beta_tool_jobsat   = ols_jobsat.params.get("ToolCountWork", 0.0)
beta_highai_sofr   = ols_sofr.params.get("HighAI", 0.0)
me_highai_threat   = marginal_effect_highai(logit_threat, df_model, "HighAI")

print("beta_highai_tool:", beta_highai_tool)
print("beta_highai_jobsat:", beta_highai_jobsat)
print("beta_tool_jobsat:", beta_tool_jobsat)
print("beta_highai_sofr:", beta_highai_sofr)
print("me_highai_threat:", me_highai_threat)

lever_impacts = []

for lever in lever_cols:
    d_highai = lever_delta_highai(logit_highai, df_model, lever)
    if pd.isna(d_highai):
        continue
    
    d_tool   = d_highai * beta_highai_tool
    d_jobsat = d_highai * beta_highai_jobsat + d_tool * beta_tool_jobsat
    d_threat = d_highai * me_highai_threat
    d_sofr   = d_highai * beta_highai_sofr
    
    lever_impacts.append({
        "lever": lever,
        "delta_highai": d_highai,
        "delta_toolcount": d_tool,
        "delta_jobsat": d_jobsat,
        "delta_aithreat": d_threat,
        "delta_sofriction": d_sofr
    })

lever_impacts_df = pd.DataFrame(lever_impacts)
lever_impacts_df.sort_values("delta_highai", ascending=False)


beta_highai_tool: 0.1173490188088265
beta_highai_jobsat: 0.24823292385989526
beta_tool_jobsat: -0.0026809811472453225
beta_highai_sofr: 0.33475379252253346
me_highai_threat: 0.059464022602383156


,lever,delta_highai,delta_toolcount,delta_jobsat,delta_aithreat,delta_sofriction
2,Lever_AIAgentsAny,0.456112,0.053524,0.113078,0.027122,0.152685
1,Lever_AIUseHigh,0.324483,0.038078,0.080445,0.019295,0.108622
0,Lever_LearnCodeAI,0.279938,0.032850,0.069402,0.016646,0.093710
7,Lever_AISentPos,0.063457,0.007447,0.015732,0.003773,0.021242
4,Lever_SQL_any,0.025563,0.003000,0.006337,0.001520,0.008557
3,Lever_NumLangsHigh,0.021579,0.002532,0.005350,0.001283,0.007224
5,Lever_HasSOAccount,0.016299,0.001913,0.004041,0.000969,0.005456
6,Lever_SOVisitHigh,-0.006783,-0.000796,-0.001682,-0.000403,-0.002271


In [59]:
def prettify_lever(name):
    mapping = {
        "Lever_LearnCodeAI": "AI Training Access",
        "Lever_AIUseHigh": "High AI Tool Usage (weekly/daily)",
        "Lever_AIAgentsAny": "Uses AI Agents",
        "Lever_NumLangsHigh": "Broad Language Stack",
        "Lever_SQL_any": "Has SQL/Data Skill",
        "Lever_HasSOAccount": "Has Stack Overflow Account",
        "Lever_SOVisitHigh": "Frequent Stack Overflow Visitor",
        "Lever_AISentPos": "Positive Attitude Toward AI"
    }
    return mapping.get(name, name)

lever_impacts_df["lever_pretty"] = lever_impacts_df["lever"].apply(prettify_lever)

# Composite impact score (tune weights if desired)
lever_impacts_df["impact_score"] = (
    1.0 * lever_impacts_df["delta_highai"] +
    0.5 * lever_impacts_df["delta_toolcount"] +
    0.5 * lever_impacts_df["delta_jobsat"] -
    0.3 * lever_impacts_df["delta_aithreat"]
)

lever_impacts_df = lever_impacts_df.sort_values("impact_score", ascending=False)
lever_impacts_df


,lever,delta_highai,delta_toolcount,delta_jobsat,delta_aithreat,delta_sofriction,lever_pretty,impact_score
2,Lever_AIAgentsAny,0.456112,0.053524,0.113078,0.027122,0.152685,Uses AI Agents,0.531276
1,Lever_AIUseHigh,0.324483,0.038078,0.080445,0.019295,0.108622,High AI Tool Usage (weekly/daily),0.377956
0,Lever_LearnCodeAI,0.279938,0.032850,0.069402,0.016646,0.093710,AI Training Access,0.326070
7,Lever_AISentPos,0.063457,0.007447,0.015732,0.003773,0.021242,Positive Attitude Toward AI,0.073914
4,Lever_SQL_any,0.025563,0.003000,0.006337,0.001520,0.008557,Has SQL/Data Skill,0.029775
3,Lever_NumLangsHigh,0.021579,0.002532,0.005350,0.001283,0.007224,Broad Language Stack,0.025135
5,Lever_HasSOAccount,0.016299,0.001913,0.004041,0.000969,0.005456,Has Stack Overflow Account,0.018985
6,Lever_SOVisitHigh,-0.006783,-0.000796,-0.001682,-0.000403,-0.002271,Frequent Stack Overflow Visitor,-0.007901


In [61]:
# Export for Tableau
lever_impacts_df.to_csv("outputs/tables/tableau_policy_lever_impacts.csv", index=False)
print("Exported tableau_policy_lever_impacts.csv")


Exported tableau_policy_lever_impacts.csv


In [63]:
import pandas as pd

df = pd.read_csv("tableau_policy_lever_impacts.csv")

# Standardize names for safety
df['lever_pretty'] = df['lever_pretty'].str.strip()

# Strategic risk mapping
risk_map = {
    "Uses AI Agents": "High Risk",
    "High AI Tool Usage (weekly/daily)": "Medium Risk",
    "AI Training Access": "Low Risk",
    "Positive Attitude Toward AI": "Low Risk",
    "Has SQL/Data Skill": "Low Risk",
    "Has Stack Overflow Account": "No Added Friction"
}

df['strategic_risk'] = df['lever_pretty'].map(risk_map)

# Safety fallback
df['strategic_risk'] = df['strategic_risk'].fillna("Low Risk")

df.to_csv("outputs/tables/tableau_policy_lever_impacts_strategic.csv", index=False)

df.head()


,lever,delta_highai,delta_toolcount,delta_jobsat,delta_aithreat,delta_sofriction,lever_pretty,impact_score,strategic_risk
0,Lever_AIAgentsAny,0.456112,0.053524,0.113078,0.027122,0.152685,Uses AI Agents,0.531276,High Risk
1,Lever_AIUseHigh,0.324483,0.038078,0.080445,0.019295,0.108622,High AI Tool Usage (weekly/daily),0.377956,Medium Risk
2,Lever_LearnCodeAI,0.279938,0.032850,0.069402,0.016646,0.093710,AI Training Access,0.326070,Low Risk
3,Lever_AISentPos,0.063457,0.007447,0.015732,0.003773,0.021242,Positive Attitude Toward AI,0.073914,Low Risk
4,Lever_SQL_any,0.025563,0.003000,0.006337,0.001520,0.008557,Has SQL/Data Skill,0.029775,Low Risk
